In [1]:
!rm -rf ClothAId_AI_Cloth_Recognition_and_Description_System_for_Visually_Impaired_BSc_Thesis
!git clone https://github.com/SotirisDimitrakoulakos/ClothAId_AI_Cloth_Recognition_and_Description_System_for_Visually_Impaired_BSc_Thesis.git

Cloning into 'ClothAId_AI_Cloth_Recognition_and_Description_System_for_Visually_Impaired_BSc_Thesis'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 206 (delta 47), reused 200 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (206/206), 135.30 KiB | 2.25 MiB/s, done.
Resolving deltas: 100% (47/47), done.


In [2]:
import pickle
from types import SimpleNamespace

efnet_training = SimpleNamespace()
efnet_validation = SimpleNamespace()
efnet_testing = SimpleNamespace()

with open('/notebooks/data/efnet_training_2.pkl', 'rb') as f:
    efnet_training.X_train, efnet_training.y_train = pickle.load(f)
with open('/notebooks/data/efnet_validation_2.pkl', 'rb') as f:
    efnet_validation.X_val, efnet_validation.y_val = pickle.load(f)
with open('/notebooks/data/efnet_testing_2.pkl', 'rb') as f:
    efnet_testing.X_test, efnet_testing.y_test = pickle.load(f)

In [3]:
# EfficientNet dataset sizes
print("EfficientNet Training size:", len(efnet_training.X_train), len(efnet_training.y_train))
print("EfficientNet Validation size:", len(efnet_validation.X_val), len(efnet_validation.y_val))
print("EfficientNet Testing size:", len(efnet_testing.X_test), len(efnet_testing.y_test))

EfficientNet Training size: 10633 7
EfficientNet Validation size: 3545 7
EfficientNet Testing size: 3544 7


In [4]:
import pandas as pd

balanced_metadata = pd.read_parquet('/notebooks/data/filtered_balanced_dataset_ef_2.parquet')

In [5]:
# Sanity print: show the first few rows
print(balanced_metadata.head())

# Optional: Print shape and info for additional sanity checks
print("\nShape:", balanced_metadata.shape)
print("\nInfo:")
print(balanced_metadata.info())

      id  gender masterCategory subCategory articleType baseColour  \
0   5271  unisex    accessories        bags   backpacks      green   
1  15653  unisex    accessories        bags   backpacks       blue   
2   7056  unisex    accessories        bags   backpacks      black   
3   4583  unisex    accessories        bags   backpacks       grey   
4  15419  unisex    accessories        bags   backpacks      black   

          season    year   usage  \
0         winter  2015.0  casual   
1           fall  2011.0  casual   
2  summer/spring  2011.0  sports   
3         winter  2015.0  casual   
4           fall  2011.0  casual   

                                  productDisplayName  
0                 Wildcraft Unisex Green Holster Bag  
1  Belkin Unisex Simple Backpack Navy Blue Backpacks  
2                Nike Unisex Trng Max Black Backpack  
3                     Wildcraft Unisex Grey Backpack  
4                 Fila Unisex Silver Black Backpacks  

Shape: (17723, 10)

Info:
<clas

In [6]:
import sys
sys.path.append('/notebooks/ClothAId_AI_Cloth_Recognition_and_Description_System_for_Visually_Impaired_BSc_Thesis/AI_infrastructure/Python_Files')

In [7]:
import importlib
import training
import efficientnet_model  # import once

importlib.reload(training)
importlib.reload(efficientnet_model)

2025-06-10 23:57:51.869014: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-10 23:57:51.869108: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-10 23:57:51.870269: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-10 23:57:51.877642: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-10 23:57:52.907231: W tensorflow/compiler/tf2

<module 'efficientnet_model' from '/notebooks/ClothAId_AI_Cloth_Recognition_and_Description_System_for_Visually_Impaired_BSc_Thesis/AI_infrastructure/Python_Files/efficientnet_model.py'>

In [ ]:
# 4. Train EfficientNet
from training import ClothingClassifierTrainer
from efficientnet_model import EfficientNetClothingClassifier


# Get number of classes for each attribute
num_classes_dict = {
    attr: len(balanced_metadata[attr].unique())
    for attr in ['masterCategory', 'subCategory', 'articleType',
                'baseColour', 'gender', 'season', 'usage']
}

# Build and train EfficientNet
effnet = EfficientNetClothingClassifier(num_classes_dict)
effnet_model = effnet.build_model()

# Set the directory where you want to save/load model and training state
save_dir = '/notebooks/data/EffNet_2/general'
save_dir_resume = '/notebooks/data/EffNet_2/general/resume'

effnet_trainer = ClothingClassifierTrainer(effnet_model, 'efficientnet', save_dir=save_dir, save_dir_resume=save_dir_resume)
history_ef = effnet_trainer.train(efnet_training.X_train, efnet_training.y_train, efnet_validation.X_val,
                                  efnet_validation.y_val, batch_size=16, epochs=50, X_test=efnet_testing.X_test, y_test=efnet_testing.y_test, resume_training=False)

effnet_trainer.save_model(save_dir)


2025-06-10 22:49:31.447680: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-06-10 22:49:31.516721: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-06-10 22:49:31.516898: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

43941136/43941136 [==============================] - 0s 0us/step
Model outputs: ['articleType', 'baseColour', 'gender', 'masterCategory', 'season', 'subCategory', 'usage']
Loss keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage']
Starting fresh training...
Encoded y_train keys and shapes:
masterCategory: (10633,)
subCategory: (10633,)
articleType: (10633,)
baseColour: (10633,)
gender: (10633,)
season: (10633,)
usage: (10633,)
Encoded y_val keys and shapes:
masterCategory: (3545,)
subCategory: (3545,)
articleType: (3545,)
baseColour: (3545,)
gender: (3545,)
season: (3545,)
usage: (3545,)
Batch 0: X shape (16, 300, 300, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Sanity Check Shapes:
X: (16, 300, 300, 3) float32
masterCategory: (16,), int64
subCategory: (16,), int64
articleType: (16,), int64
baseColour: (16,), int64
gender: (16,), int64
season: (16,), int64
usage: (

2025-06-10 22:49:44.635488: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


  1/665 [..............................] - ETA: 1:45:52 - loss: 17.5774 - articleType_loss: 4.4806 - baseColour_loss: 2.9343 - gender_loss: 1.6681 - masterCategory_loss: 1.5291 - season_loss: 1.3519 - subCategory_loss: 3.3913 - usage_loss: 2.2222 - articleType_accuracy: 0.0625 - baseColour_accuracy: 0.0625 - gender_accuracy: 0.1250 - masterCategory_accuracy: 0.1875 - season_accuracy: 0.2500 - subCategory_accuracy: 0.0625 - usage_accuracy: 0.2500

2025-06-10 22:49:45.857257: I external/local_xla/xla/service/service.cc:168] XLA service 0xcc7b750 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-06-10 22:49:45.857326: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA RTX A4000, Compute Capability 8.6
I0000 00:00:1749595785.943413     127 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Batch 527: X shape (16, 300, 300, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
665/665 [==============================] - ETA: 0s - loss: 7.9369 - articleType_loss: 2.4978 - baseColour_loss: 2.0958 - gender_loss: 0.6227 - masterCategory_loss: 0.2119 - season_loss: 0.8687 - subCategory_loss: 1.0043 - usage_loss: 0.6358 - articleType_accuracy: 0.5305 - baseColour_accuracy: 0.3304 - gender_accuracy: 0.7944 - masterCategory_accuracy: 0.9576 - season_accuracy: 0.6221 - subCategory_accuracy: 0.7849 - usage_accuracy: 0.8071Batch 0: X shape (16, 300, 300, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Batch 0: X shape (16, 300, 300, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Batch 1: X shape (16, 300, 300, 3), y keys: ['masterCategory', 'subCategory', 'articleType

In [10]:
#Resume Training

# 4. Train EfficientNet
from training import ClothingClassifierTrainer
from efficientnet_model import EfficientNetClothingClassifier

  
# Get number of classes for each attribute
num_classes_dict = {
    attr: len(balanced_metadata[attr].unique())
    for attr in ['masterCategory', 'subCategory', 'articleType',
                'baseColour', 'gender', 'season', 'usage']
}

# Build and train EfficientNet
effnet = EfficientNetClothingClassifier(num_classes_dict)
effnet_model = effnet.build_model()

# Set the directory where you want to save/load model and training state
save_dir = '/notebooks/data/EffNet_2/general'
save_dir_resume = '/notebooks/data/EffNet_2/general/resume'

effnet_trainer = ClothingClassifierTrainer(effnet_model, 'efficientnet', save_dir=save_dir, save_dir_resume=save_dir_resume)
history_ef = effnet_trainer.train(efnet_training.X_train, efnet_training.y_train, efnet_validation.X_val,
                                  efnet_validation.y_val, batch_size=16, epochs=50, X_test=efnet_testing.X_test, y_test=efnet_testing.y_test, resume_training=True)

effnet_trainer.save_model(save_dir)


Model outputs: ['articleType', 'baseColour', 'gender', 'masterCategory', 'season', 'subCategory', 'usage']
Loss keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage']
Resuming training...
Loaded weights from /notebooks/data/EffNet_2/general/resume/efficientnet_final_weights.weights.h5
Resuming from epoch 50
Encoded y_train keys and shapes:
masterCategory: (10633,)
subCategory: (10633,)
articleType: (10633,)
baseColour: (10633,)
gender: (10633,)
season: (10633,)
usage: (10633,)
Encoded y_val keys and shapes:
masterCategory: (3545,)
subCategory: (3545,)
articleType: (3545,)
baseColour: (3545,)
gender: (3545,)
season: (3545,)
usage: (3545,)
Batch 0: X shape (16, 300, 300, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Sanity Check Shapes:
X: (16, 300, 300, 3) float32
masterCategory: (16,), int64
subCategory: (16,), int64
articleType: (16,), int64
baseColour: (16,), int64


ValueError: No loss keys found in training history. Got keys: []

In [ ]:
# Fine-Tuning

import json
from tensorflow.keras.models import model_from_json

# Get number of classes for each attribute
num_classes_dict = {
    attr: len(balanced_metadata[attr].unique())
    for attr in ['masterCategory', 'subCategory', 'articleType',
                'baseColour', 'gender', 'season', 'usage']
}

# 1. Load model architecture from .json file
model_name = 'efficientnet'
save_dir = '/notebooks/data/EffNet_2/general'  # path to previously saved model
save_dir_resume = '/notebooks/data/EffNet_2/fine_tuned/resume'


effnet = EfficientNetClothingClassifier(num_classes_dict)
model_arch = effnet.build_model()

# 2. Load model weights and label encoders
model, label_encoders = ClothingClassifierTrainer.load_model(
    model_name=model_name,
    model_arch=model_arch,
    save_dir=save_dir,
    best_weights=True
)

# 3. Fine-tune the model (unfreeze layers and compile)
model = effnet.fine_tune(model)

# 4. Create a new trainer for fine-tuning
trainer = ClothingClassifierTrainer(model, model_name, save_dir='/notebooks/data/EffNet_2/fine_tuned', save_dir_resume=save_dir_resume)
trainer.label_encoders = label_encoders

# 5. Train the fine-tuned model
history_ef_ft = trainer.train(
    efnet_training.X_train,
    efnet_training.y_train,
    efnet_validation.X_val,
    efnet_validation.y_val,
    batch_size=16,
    epochs=20,
    resume_training=False,
    fit_label_encoders=False
)

trainer.save_model('/notebooks/data/EffNet_2/fine_tuned')

Model outputs: ['articleType', 'baseColour', 'gender', 'masterCategory', 'season', 'subCategory', 'usage']
Loss keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage']
✅ Loaded best weights from /notebooks/data/EffNet_2/general/efficientnet_best_weights.weights.h5
Starting fresh training...
Encoded y_train keys and shapes:
masterCategory: (10633,)
subCategory: (10633,)
articleType: (10633,)
baseColour: (10633,)
gender: (10633,)
season: (10633,)
usage: (10633,)
Encoded y_val keys and shapes:
masterCategory: (3545,)
subCategory: (3545,)
articleType: (3545,)
baseColour: (3545,)
gender: (3545,)
season: (3545,)
usage: (3545,)
Batch 0: X shape (16, 300, 300, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Sanity Check Shapes:
X: (16, 300, 300, 3) float32
masterCategory: (16,), int64
subCategory: (16,), int64
articleType: (16,), int64
baseColour: (16,), int64
gender: (16,), int

2025-06-11 00:32:14.577427: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


  2/665 [..............................] - ETA: 41s - loss: 2.3980 - articleType_loss: 0.2424 - baseColour_loss: 1.1134 - gender_loss: 0.2049 - masterCategory_loss: 0.0186 - season_loss: 0.5210 - subCategory_loss: 0.0560 - usage_loss: 0.2416 - articleType_accuracy: 0.9375 - baseColour_accuracy: 0.5625 - gender_accuracy: 0.9062 - masterCategory_accuracy: 1.0000 - season_accuracy: 0.7812 - subCategory_accuracy: 1.0000 - usage_accuracy: 0.9375    

2025-06-11 00:32:15.711736: I external/local_xla/xla/service/service.cc:168] XLA service 0xca288a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-06-11 00:32:15.711783: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA RTX A4000, Compute Capability 8.6
I0000 00:00:1749601935.779204    1994 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Batch 90: X shape (16, 300, 300, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
665/665 [==============================] - ETA: 0s - loss: 2.6842 - articleType_loss: 0.3334 - baseColour_loss: 1.0855 - gender_loss: 0.2558 - masterCategory_loss: 0.0177 - season_loss: 0.6310 - subCategory_loss: 0.1125 - usage_loss: 0.2482 - articleType_accuracy: 0.9051 - baseColour_accuracy: 0.6494 - gender_accuracy: 0.9064 - masterCategory_accuracy: 0.9959 - season_accuracy: 0.7242 - subCategory_accuracy: 0.9682 - usage_accuracy: 0.9109Batch 0: X shape (16, 300, 300, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Batch 0: X shape (16, 300, 300, 3), y keys: ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'gender', 'season', 'usage'], y[0] shape: (16,)
Batch 1: X shape (16, 300, 300, 3), y keys: ['masterCategory', 'subCategory', 'articleType'

In [ ]:
#Resume Fine-Tuning
# Fine-Tuning

import json
from tensorflow.keras.models import model_from_json

# Get number of classes for each attribute
num_classes_dict = {
    attr: len(balanced_metadata[attr].unique())
    for attr in ['masterCategory', 'subCategory', 'articleType',
                'baseColour', 'gender', 'season', 'usage']
}

# 1. Load model architecture from .json file
model_name = 'efficientnet'
save_dir = '/notebooks/data/EffNet_2/general'  # path to previously saved model
save_dir_resume = '/notebooks/data/EffNet_2/fine_tuned/resume'

with open(f'{save_dir}/{model_name}_architecture.json', 'r') as f:
    model_json = f.read()
model_arch = model_from_json(model_json)

# 2. Load model weights and label encoders
model, label_encoders = ClothingClassifierTrainer.load_model(
    model_name=model_name,
    model_arch=model_arch,
    save_dir=save_dir,
    best_weights=True
)

# 3. Fine-tune the model (unfreeze layers and compile)
effnet = EfficientNetClothingClassifier(num_classes_dict)
model = effnet.fine_tune(model)

# 4. Create a new trainer for fine-tuning
trainer = ClothingClassifierTrainer(model, model_name, save_dir='/notebooks/data/EffNet_2/fine_tuned', save_dir_resume=save_dir_resume)
trainer.label_encoders = label_encoders

# 5. Train the fine-tuned model
history_ef_ft = trainer.train(
    efnet_training.X_train,
    efnet_training.y_train,
    efnet_validation.X_val,
    efnet_validation.y_val,
    batch_size=16,
    epochs=20,
    resume_training=True
)

trainer.save_model('/notebooks/data/EffNet_2/fine_tuned')